Based on "warp_supernova_select" mixed up with warp_template


- Look at things class by class.
- Define subset of BTS supernovae below some redshift cat, with type and minimal data quality.
- For each SN and template compbo, find the warp coefficients that best match. Store these if "successful"
- 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import pickle
import json
from astropy.cosmology import Planck13 as cosmo
import pymongo
import sncosmo
from warptemplate import get_ztftable_from_ampel, get_warpedTimeSeriesModel, get_template_correction
from scipy.stats.distributions import chi2

In [ ]:
import warnings
from iminuit.util import IMinuitWarning

warnings.filterwarnings("ignore", category=IMinuitWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
client = pymongo.MongoClient()
db = client.bts_ipacfp_strictbase    # Only final lc. try bts_ipacfp_strictbase_full for all alerts

Lessons learned:

From first run through, examination of the original bts classes. Which to determine which can be used, which should be added to others and what redshift limit to use.
- Ca-rich: Single, skip class.
- ILRT: Only two, skip class.
- SLSN-II: zrange 0.07, 0.17
- SN Ia-91 bg: default z ok (could increase to 0.05 to add some more, but biased)
- SN Ia-91T: z range 0.03 < 0.08,
- SN Ia-CSM: skip, fluctuations anyway at late phases.
- SN Ia-pec: <0.06 (0.055), combine with other weird?
- SN Iax: <0.055. Keep alone or add to Ia-pec?
- SN Ib: Default is fine.
- SN Ibn: Combine with Ib?
- SN Ic: Default is fine.
- SN Ic-BL: Combine with Ic.
- SN Icn: Single, skip class.
- SN II: Default is fine (could increase range?)
- SN II-pec: add to SNII.
- SN IIL: add to SN II
- SN IIP: Keep as is, but could also include in general SNII
- SN Ib/c: Keep range, but then also add Ib/Ic for parent class?
- TDE: Could work, but better models exist. So skip for now.

  Some files not found, like SN IIn - why?
  What about the SNIa classes? (Iax, 91bg, 91T ? Should these start from salt templates or other keep now?)

In [ ]:
# References warpclasses 
classnbr = 3

In [ ]:
# This is the classes which we eventually expect will be made into 
# warptemplate classes. 
# Todo: some BTS classes mising, like SLSN-I & SN IIn. 
#       Probably for some good reason, but need to verify for paper.
warpclasses = [
    'SN Ib', 'SN Ia-91bg', 'SN II', 'SN Ia-91T',
    'SN Ib/c', 'SN Ia-pec', 'SN IIP', 'SN Iax', 'SN Ic',
    'SLSN-II']
# Some of these will be umbrella classes and we will combine btsfits files:
parentclasses = {
    'SN Ic':['SN Ic', 'SN Ic-BL', 'SN Ic-pec', 'SN Icn'], 
    # Note that "/" not used in filenames
    'SN Ib/c':['SN Ibc', 'SN Ib', 'SN Ic', 'SN Ibn', 'SN Ic-BL', 'SN Ic-pec', 'SN Icn'], 
    'SN II': ['SN II','SN IIP', 'SN II-pec', 'SN IIL']
}
# Redshift ranges for particular classes
zclass = {
    'SLSN-II': [0.07, 0.17],
    'SN Ia-91bg': [0.01, 0.055],
    'SN Ia-91T': [0.03, 0.08],
    'SN Ia-pec': [0.01, 0.055], 
    'SN Iax': [0.01, 0.055], 
}

In [ ]:
# So, what are our recommendeded limits?
# These are the things we can try
# Ok, fix these ... seems like we cannot really increase the requirements 
# and keep sufficient data. Instead, think of a ranking of data
# and use the best fits.
zlim = [0.01,0.04]
min_bands= 2
min_peakndof = 6
require_peak_good = False
# Fixed params
fdir = '/Users/jnordin/data/models/sncosmo/'
peakfit = True          # Only use data round peak to estimate fit quality 
aiceval = False        # Rank fits based on aic (True) or chi/dof (False)
# Settings for template running - optimized previoiusly - where! ... accept for now?
phaserange = 60
intdisp =  0.1
fracdisp = 0
peak_chicut = 1.5

# % error floor - is this what we want? could also have some fixed floor etc. try this
errfloor = 0.05


In [ ]:
if warpclasses[classnbr] in zclass:
    zlim = zclass[warpclasses[classnbr]]

### Define true match
We need a mapping to check whether a given SNcosmo class should be considered a true fit to a given BTS def.

In [ ]:
# For each warpclass, define which sncosmo templates to be 
# considered "true", if only including such templates as models.
bts_truth_map = {
    'SN Ib': ['SN Ic', 'SN Ib/c', 'SN Ib', 'SN Ic-BL'], 
    'SN Ia-91bg': ['SN Ia'], 
    'SN II': ['SN II', 'SN IIL', 'SN IIl', 'SN IIp', 'SN IIB','SN IIb', 'SN IIP', 'SN IIn', 'SN IIN'], 
    'SN Ia-91T': ['SN Ia'], 
    'SN Ib/c': ['SN Ic', 'SN Ib/c', 'SN Ib', 'SN Ic-BL'], 
    'SN Ia-pec': ['SN Ia'], 
    'SN IIP': ['SN II', 'SN IIp' 'SN IIP'], 
    'SN Iax': ['SN Ia'], 
    'SN Ic': ['SN Ic', 'SN Ib/c', 'SN Ib', 'SN Ic-BL'],
    'SLSN-II': ['SN II', 'SN IIL', 'SN IIl', 'SN IIp', 'SN IIB','SN IIb', 'SN IIP', 'SN IIn', 'SN IIN', 'PopIII'],     # No templates available, using these for now
    'SN IIn': ['SN II', 'SN IIL', 'SN IIl', 'SN IIp', 'SN IIB','SN IIb', 'SN IIP', 'SN IIn', 'SN IIN'], 
}

### Start processing

In [ ]:
def combine_typefits( dictlI, dictlII ):
    """
    Combine the internal dicts of two lists of dicts.
    """
    for templatename, datalist in dictlII.items():
        dictlI[templatename].extend( datalist )
    return dictlI 

In [ ]:
typefitdata = None
if warpclasses[classnbr] in parentclasses:
    toread = parentclasses[warpclasses[classnbr]]
else:
    toread = [warpclasses[classnbr]]

for readclass in toread:
    rootname = 'btsfits_{}_{}_{}_{}_{}'.format(
        readclass, str(phaserange), str(peak_chicut), str(fracdisp), str(intdisp)
    )
    fname = fdir+rootname+'.json'
    # Open data
    with open(fname, "r") as infile:
        if typefitdata is None:
            typefitdata = json.loads(infile.read())
        else:
            typefitdata = combine_typefits( 
                typefitdata, json.loads(infile.read()) 
            )
    
    print(readclass, len(typefitdata))

In [ ]:
def get_salt_cosmofit( fitlist,
                        chidofmax=3,    # Max chi/dof to be counted as good fit
                        truetypes = []
                     ):
    """
    Go through the input list of saltfits. Return a list of the subset which fulfills
    selection criteria as specified. The idea is that the provided list of templates
    are "good enough" for template construction.

    Parameters:
    - 
    determine whether the fit is good and whether it 
    could be considered for cosmology.

    If the model class included in truetypes, classification set to correct.

    Nothing returned for failed fits.

    Return a pandas df
    """

    outd = []

    for snfit in fitlist:
        if not snfit['success']:
            continue
        snd = {
            k:snfit[k] for k in ['z', 'chisq', 'ndof', 'peakmag', 'chidof', 'id', 'nbr_bands', 'ndet','class',
                                 'peakchi','peakdet','earlydet','peakbands','peak_good']
        }
        fitp = snfit['ndet']-snfit['ndof']
        snd['aic'] = 2 * fitp + snd['chisq'] # 4 dof, ln L = - 0.5 chi 
        snd['peakndof'] = snfit['peakdet']-fitp

        if peakfit:
            if not snd['peakndof']>0:
                continue
            snd['peakchisqdof'] = snfit['peakchi']/snd['peakndof']
            snd['peakaic'] = 2 * fitp + snfit['peakchi'] # 4 dof, ln L = - 0.5 chi
            snd['chidof'] = snfit['peakchi']/snd['peakndof']
            snd['aic'] = snd['peakaic']
        else:
            snd['chidof'] = snfit['chidof']

            
        # Check for good, correct fit
        snd['correct'] = False
        if snd['class'] in truetypes:
            snd['correct'] = True
        #snd['goodfit'] = False
        #if ( (-4 < snfit['x1'] < 4) and (-0.5 < snfit['c'] < 1.5) and (snd['chidof'] < chidofmax) 
        #    and (snfit['errors']['x1'] < 2.) and (snfit['errors']['c'] < 1.) ):
        #    snd['goodfit'] = True
        if (snd['chidof'] < chidofmax):
            snd['goodfit'] = True
        else:
            snd['goodfit'] = False

        outd.append(snd)

    return pd.DataFrame.from_dict(outd)


def get_timeseries_goodfit( fitlist,
                        chidofmax=3,    # Max chi/dof to be counted as good fit
                        truetypes = [],                           
                          ):
    """
    Go through the input list of an sncosmo timeseries model with added dust. 
    For each, determine whether the fit is good.

    Nothing returned for failed fits.

    Assumes fit made with amplitude, rv and ebv. Should probably redo without fitting both ... 

    Return a pandas df
    """

    outd = []

    for snfit in fitlist:
        if not snfit['success']:
            continue
        snd = {
            k:snfit[k] for k in ['z', 'chisq', 'ndof', 'peakmag', 'chidof', 'id', 'nbr_bands', 'ndet','class',
                                 'peakchi','peakdet','earlydet','peakbands','peak_good']
                                 
        }
        fitp = snfit['ndet']-snfit['ndof']
        snd['aic'] = 2 * fitp + snd['chisq'] # ln L = - 0.5 chi 

        snd['peakndof'] = snfit['peakdet']-fitp

        if peakfit:
            if not snd['peakndof']>0:
                continue
            snd['peakchisqdof'] = snfit['peakchi']/snd['peakndof']
            snd['peakaic'] = 2 * fitp + snfit['peakchi'] # 4 dof, ln L = - 0.5 chi
            snd['chidof'] = snfit['peakchi']/snd['peakndof']
            snd['aic'] = snd['peakaic']
        else:
            snd['chidof'] = snfit['chidof']


        
        # Check for good fit (note that r_v is typically not find and should be trivially fulfilled)
        #if ( ( -0.3 < snfit['hostebv'] < 2.5) and (0.1 < snfit['hostr_v'] < 5) and (snd['chidof'] < chidofmax) ):
        #    snd['goodfit'] = True
        #else:
        #    snd['goodfit'] = False

        if (snd['chidof'] < chidofmax):
            snd['goodfit'] = True
        else:
            snd['goodfit'] = False

            
        if snd['class'] in truetypes:
            snd['correct'] = True
        else:
            snd['correct'] = False

        outd.append(snd)

    return pd.DataFrame.from_dict(outd)


In [ ]:
modelfits = {}
for modelname, modeldata in typefitdata.items():
    if re.search('salt', modelname):
        df = get_salt_cosmofit( modeldata, truetypes=bts_truth_map[warpclasses[classnbr]] )
    else:
        df = get_timeseries_goodfit( modeldata, truetypes=bts_truth_map[warpclasses[classnbr]] )
    df.insert(0,'model', modelname) 
    
    modelfits[modelname] = df

In [ ]:
# Combine to one df
dfall = pd.concat( modelfits.values(), ignore_index=True )

In [ ]:
dfall.shape

In [ ]:
# Ok, so this is where we skip directly to the warp stage. We iterate through each row:
# - Fit a warp and plot a warp template, looking at the parameer for this.
# - Possibly we do some quality cut arleady now and we do not include template of different classes.
# - as a start, only run for the target objects from Suleyman ...

In [ ]:
# As testing, we might only do the Suleyman troublesome set to speed things up
onlydo = [
    'ZTF18aadsuxd', 'ZTF19abkfqqp', 'ZTF18abfcmjw', 'ZTF19abzqwpr', 'ZTF18acrxpzx', 'ZTF19acxpuql',
    'ZTF19aalouag', 'ZTF20aapjiwl', 'ZTF19aanjipu', 'ZTF20aatvdwr', 'ZTF19aapfmki', 'ZTF20aayxdse',
    'ZTF19aapwnmb', 'ZTF21aaiaqhh', 'ZTF19abamqxo', 'ZTF21aamobbb', 'ZTF19abcegvm', 'ZTF21aapdumx',
    'ZTF21aapfdqw', 'ZTF21aazcgof', 'ZTF21aapffqd', 'ZTF21abkoccb', 'ZTF21abmjgwf', 'ZTF22aahftli' 
]
fitprop = ['t0', 'amplitude', 'hostebv']
onlydo = []

In [ ]:
flux_frac_disperson = 0.0

In [ ]:
done = []
for k, row in dfall.iterrows():

    if k%100==0:
        print('at {} of {}'.format(k,dfall.shape[0]))
    
    if re.search('salt', row['model']):
        continue
    if len(onlydo)>0 and not row['id'] in onlydo:
        continue
    # Check whether it is the correct type
    if not row['class'] in bts_truth_map[warpclasses[classnbr]]:
        continue
    # Check whether lc fulfill criteria
    if row['nbr_bands']<min_bands:
        continue
    if row['peakdet']<min_peakndof:
        continue


    # Ok, so now we try to do the fit here
#    print('fitting {} with {}'.format(row['id'],row['model']))

    # Get SN table
    # Lets grab some data, starting with the target SN
    tab = get_ztftable_from_ampel( row['id'], db, 
                              redshift=float(row['z']), 
                              include_sigma=5,
                              type = row['class'] )
    # Add errorfloor
    tab['fluxerr'] = np.sqrt((errfloor*np.mean(tab['flux']))**2+(tab['fluxerr'])**2)
    tab['fluxerr'] = np.sqrt((flux_frac_disperson*tab['flux'])**2+(tab['fluxerr'])**2)   

    # Get correction factors 
    mdict = get_template_correction( tab, row['model'], z=float(row['z']), 
                                    pull_cut=10, plot_dir=None, 
                                    spline_lam=0.001,
                                   require_phasecoverage=False)

    if not mdict['success']:
#        print('failed, continue')
        continue


    wm = get_warpedTimeSeriesModel(
                name=row['id']+'_'+row['model'], 
                original_template_name=row['model'], 
                warpdata=mdict,
                z=float(row['z']),
                original_template_version=None
            )

    try:
        wresult, wfitted_model = sncosmo.fit_lc(
                        tab, wm,
                        fitprop,  # parameters of model to vary
                    )
    except RuntimeError:
#        print('fit failed - skipping')
        continue

    # We need some way of verifying that the template peak fit time is not before or after the data
    if (tab['time'].min()>wresult['parameters'][1] or tab['time'].max()<wresult['parameters'][1]):
#        print('peak outside data range - skippin')
        continue
    if ( (tab['time'].min()-wresult['parameters'][1])<(wm.source.minphase()-3) ):
#        print('first detection prior to template start - we do not allow this')
        continue
    # We do not want cases where the first detection is more than ten days after peak
    if ( (tab['time'].min()-wresult['parameters'][1])>5 ):
#        print('first detection prior more than five days after peak - we do not allow this')
#        break
        continue

    # Fit done - start saving
    fitres = dict(row)
    fitres['mdict'] = mdict
    fitres['wresult'] = wresult

#    sncosmo.plot_lc(tab, wfitted_model, fname='/Users/jnordin/tmp/warptest/{}_{}_warp.png'.format(row['id'],row['model']))

    
    # Also fit the raw version of this for comparison?
#    m = sncosmo.Model(source=row['model'],
#                        effects=[sncosmo.CCM89Dust()],
#                        effect_names=['host'],
#                        effect_frames=['rest'])
#    m.set(hostr_v = 3.1)
#    m.set(z=row['z'])
#    result, fitted_model = sncosmo.fit_lc(
#                    tab, m,
#                    fitprop,  # parameters of model to vary
#                )       
#    sncosmo.plot_lc(tab, fitted_model, fname='/Users/jnordin/tmp/warptest/{}_{}_og.png'.format(row['id'],row['model']))



    done.append( fitres )

In [ ]:
def apply_floor_and_normalize(p, floor):
    p = np.array(p, dtype=float)
    n = len(p)
    
    if floor * n > 1:
        raise ValueError("Floor too large to maintain sum=1")
    
    # Start by assigning the floor everywhere
    result = np.full(n, floor)
    
    # Remaining probability mass to distribute
    remaining = 1 - floor * n
    
    # Original excess above floor
    excess = np.maximum(p - floor, 0)
    
    if excess.sum() > 0:
        result += remaining * (excess / excess.sum())
    else:
        # If all values were below floor, distribute evenly
        result += remaining / n
    
    return result

In [ ]:
peakprobs = []
probcut = 0.01
min_draw_prob = 0.01
min_templates = 3
template_collection = []
for sn in set( [ f['id'] for f in done] ):

    # Get fit quality for this sn
    fits_here = [
        { 'model': f['model'], 'chi':f['wresult']['chisq'], 'peakdet':f['wresult']['ndof'], 
         'z': f['z'], 'peakmag': f['peakmag'], 
         'sf': chi2.sf(f['wresult']['chisq'],f['wresult']['ndof']) }
            for f in done 
        if f['id']==sn
        
    ]
    df_here = pd.DataFrame.from_dict(fits_here)

    peakprob = df_here['sf'].max()
    mask_peakprob = (df_here['sf']==peakprob)
    print('SN: {} Peak prob {} for template {}'.format(sn, peakprob, df_here.loc[mask_peakprob,'model'].iloc[0]))


    prob_mask = (df_here['sf']>probcut)
    good_templates = None
    if sum(prob_mask)>=min_templates:
        print('... enough within prob limit, using these')
        df_subset = df_here.loc[prob_mask,:].copy()
        good_templates = True
    else:
        print('not enough good models, want to a finite number. Plotting these.')
        df_subset = df_here.nlargest(min_templates, "sf")
        good_templates = False
    df_subset.loc[:,'draw_prob'] = apply_floor_and_normalize( df_subset['sf'], min_draw_prob )
#    for mod in list(df_subset['model']):
#        print('open /Users/jnordin/tmp/warptest/{}_{}_warp.png'.format(sn, mod))

    # So, now we have the list of templates to be used for this SN. Somehow record this?
    # Stupid to go through all of the cases again, but why not ...
    # We will save this as a per "warpclass" collection. So when creating from this collection one draws
    # one object from the list, corresponding to one SN. That is stored as a dict, labeled by model, with probability as one of the main keys
    # (cannot have probability as raw). But they might not be unique? better not...
    sndict = {'ztfid':sn, 'good_fit':good_templates,'warpmodels':[]}
    for i, snrow in df_subset.iterrows():
        info = dict(snrow)
        modelfit = None
        for warpfit in done:
            if not (warpfit['model']==info['model'] and warpfit['id']==sn):
                continue
            modelfit = warpfit
            break
        info['mdict'] = modelfit['mdict']
        sndict['warpmodels'].append( info )
        
    template_collection.append( sndict )


In [ ]:
len(template_collection)

In [ ]:
#dfsum = pd.DataFrame.from_dict( snstore )

In [ ]:
#dfsum['absmag'] = dfsum['peakmag'] - cosmo.distmod(dfsum['z']).value

In [ ]:
storefile = '/Users/jnordin/data/models/sncosmo/warpmod/warpcoeffs_'+re.sub(r'/', '', warpclasses[classnbr])+'.pkl'

In [ ]:
with open(storefile, 'wb') as file:
    pickle.dump(template_collection, file)

In [ ]:
info['draw_prob']

In [ ]:
info